# W02b - Topic Modeling (BoW) & ML Regression Training

In [1]:
### Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import gensim
import pickle
import time
from text_operations import manipulate_text, translate_to_english
from gensim.utils import simple_preprocess
from gensim.test.utils import datapath
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',None)
num_topics = 25

In [2]:
### Load the salary dataset with the correct date, plus the country coefficient dataset
salaries = pd.read_csv('Salaries_v4_202412161100_enhanced.csv')
salaries = salaries.drop('Unnamed: 0', axis=1)
countries_coef = pd.read_csv('countries_salary_coef.csv')

## Dataset Information

In [3]:
### Number of rows (salaries) and columns
print("SALARY DATASET SHAPE -> ROWS: {}  COLUMNS: {}".format(salaries.shape[0], salaries.shape[1]))

SALARY DATASET SHAPE -> ROWS: 19554  COLUMNS: 39


In [4]:
### First few rows of the dataset
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd,country_coef,title_code,title_slr_coef,title_seniority,title_seniority_slr_coef,position_code,position_slr_coef,position_seniority,position_seniority_slr_coef,employment_type_coef,emp_type,employment_type_slr_coef,highest_edu_deg_name,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_code,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_code,company_slr_coef,main_skill_code,main_skill_slr_coef_avg,main_skill_slr_coef_max
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000,1.00,WC68,0.488,NO_SENIORITY,0.488,WC66,0.492,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,MASTERS,0.75,0.770,EDM17,0.796,0.796,UNCT,0.368,"MS01,MS02,MS03,MS04,MS05,MS06,MS07,MS11,MS17,M...",0.872,0.944
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000,1.00,UNCT,0.319,ASSOCIATE,0.319,UNCT,0.329,ASSOCIATE,0.340,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,NO_DEGREE,0.200,0.200,NO_COMPANY,0.200,"MS00,MS13,MS15,MS16,MS28,MS30,MS31,MS33,MS34,M...",0.646,0.731
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191,0.61,WC76,0.399,NO_SENIORITY,0.488,WC67,0.400,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,"EDM14,EDM30,EDM38,EDM53",0.779,0.823,NO_COMPANY,0.200,"MS00,MS16,MS40,MS42,MS43,MS45,MS62",0.774,0.844
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000,1.00,WC36,0.955,NO_SENIORITY,0.488,WC32,0.959,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM14,0.823,0.823,CMP01,0.845,"MS20,MS42,MS43",0.873,0.944
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000,1.00,WC07,0.503,NO_SENIORITY,0.488,WC06,0.516,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM19,0.655,0.655,CMP00,0.593,"MS03,MS08,MS12,MS14,MS19",0.769,0.870


In [5]:
### All features' info in the salary dataset
salaries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19554 entries, 0 to 19553
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             19554 non-null  object 
 1   found_country                  19554 non-null  object 
 2   title                          19542 non-null  object 
 3   position                       19554 non-null  object 
 4   employment_type                12397 non-null  object 
 5   company                        16936 non-null  object 
 6   company_score                  15739 non-null  float64
 7   edu_degrees                    19000 non-null  object 
 8   edu_degrees_major              18623 non-null  object 
 9   working_year                   19554 non-null  int64  
 10  education_score                14956 non-null  float64
 11  ms_counts                      19554 non-null  int64  
 12  skill_counts                   19554 non-null 

In [6]:
### Stats for numerical features of the dataset
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd,country_coef,title_slr_coef,title_seniority_slr_coef,position_slr_coef,position_seniority_slr_coef,employment_type_coef,employment_type_slr_coef,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_slr_coef,main_skill_slr_coef_avg,main_skill_slr_coef_max
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981,0.966054,0.452360,0.511637,0.456774,0.550829,0.651904,0.701566,0.612090,0.705434,0.695443,0.714984,0.517684,0.784424,0.884132
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433,0.109942,0.145708,0.074651,0.148300,0.087546,0.426533,0.382863,0.178708,0.128333,0.147528,0.158501,0.206434,0.083624,0.089182
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000,0.610000,0.102000,0.200000,0.115000,0.248000,0.100000,0.200000,0.100000,0.200000,0.200000,0.200000,0.181000,0.569000,0.569000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000,1.000000,0.350000,0.488000,0.338000,0.518000,0.100000,0.200000,0.500000,0.657000,0.591000,0.603000,0.368000,0.712000,0.833000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000,1.000000,0.484000,0.488000,0.492000,0.518000,0.990000,1.000000,0.500000,0.657000,0.732000,0.749000,0.541000,0.801000,0.894000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000,1.000000,0.539000,0.488000,0.565000,0.518000,0.990000,1.000000,0.750000,0.770000,0.796000,0.806000,0.676000,0.856000,0.964000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.990000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Preprocessing

In [7]:
### Process the null titles and companies to become empty strings
salaries['title'] = salaries['title'].apply(lambda x: '' if type(x) == float else x)
salaries['company'] = salaries['company'].apply(lambda x: '' if type(x) == float else x)

In [8]:
### Convert all textual values into list vectors
### All characters must be lowercase and several specific characters and substring should be converted into meaningful ones
salaries_text_processed = (salaries['title'] + ' ' + salaries['main_skills'] + ' ' + salaries['skills']).str.lower()
s_time = time.time()
for i in range(len(salaries_text_processed)):
    text = salaries_text_processed.iloc[i]
    manipulated_text = manipulate_text(text)
    translated_text = translate_to_english(manipulated_text)
    salaries_text_processed.iloc[i] = translated_text
salaries_text_processed = salaries_text_processed.apply(lambda x: x.split(' '))
print(">>> TOTAL TEXT PREPROCESSING TIME: {:.4f} seconds".format(time.time()-s_time))

>>> TOTAL TEXT PREPROCESSING TIME: 5.0634 seconds


In [9]:
### Check the vector structure of a randomly selected row
rand_row = random.randint(0,len(salaries_text_processed)-1)
print("### ROW {} ###\n{}".format(rand_row,salaries_text_processed.iloc[rand_row]))

### ROW 17746 ###
['underwriter', 'state', 'farm', 'analytical', 'skills', 'attention', 'detail', 'classroom', 'management', 'coaching', 'mentoring', 'communication', 'creativity', 'critical', 'thinking', 'customer', 'service', 'data', 'analysis', 'employee', 'engagement', 'financial', 'management', 'instructional', 'design', 'instructional', 'skills', 'interpersonal', 'skills', 'leadership', 'management', 'multitasking', 'presentation', 'skills', 'problem', 'solving', 'project', 'management', 'public', 'speaking', 'relationship', 'building', 'student', 'engagement', 'time', 'management', 'training', 'writing', 'adaptation', 'analytical', 'skills', 'attention', 'detail', 'classroom', 'management', 'coaching', 'collaborative', 'problem', 'solving', 'communication', 'creativity', 'innovation', 'critical', 'thinking', 'customer', 'engagement', 'customer', 'satisfaction', 'customer', 'service', 'customer', 'success', 'data', 'analysis', 'diversity', 'inclusion', 'elearning', 'easily', 'ada

In [10]:
### Create a dictionary, containing the number of times a word appears in the training set
salaries_text_processed_sr = pd.Series(salaries_text_processed)
s_time = time.time()
dictionary = gensim.corpora.Dictionary(salaries_text_processed_sr)
first_words = []
count = 0
### The first 75 words are shown in the output below
print("#### {} WORDS FOUND IN THE DICTIONARY ####".format(len(dictionary)))
print("------------- THE FIRST 75 WORDS -------------")
for k, v in dictionary.iteritems():
    first_words.append(v)
    count += 1
    if count > 74:
        break
for i in range(25):
    print("{:3} - {:24} | {:3} - {:24} | {:3} - {:24}".format(
        i, first_words[i], i+25, first_words[i+25], i+50, first_words[i+50]))
print(">>> TIME TAKEN FOR DICTIONARY CREATION: {:.4f} seconds".format(time.time()-s_time))

#### 15375 WORDS FOUND IN THE DICTIONARY ####
------------- THE FIRST 75 WORDS -------------
  0 - ajax                     |  25 - sheets                   |  50 - techniques              
  1 - audible                  |  26 - software                 |  51 - training                
  2 - c                        |  27 - sql                      |  52 - trends                  
  3 - cascading                |  28 - style                    |  53 - website                 
  4 - cplusplus                |  29 - xml                      |  54 - agile                   
  5 - css                      |  30 - associate                |  55 - analysis                
  6 - data                     |  31 - building                 |  56 - api                     
  7 - database                 |  32 - communication            |  57 - attention               
  8 - developer                |  33 - converse                 |  58 - automated               
  9 - development              |  

## Bag of Words

In [11]:
### For each document, create an individual dictionary reporting how many words and how many times those words appear.
### Save this to 'bow_corpus'
bow_corpus = [dictionary.doc2bow(s) for s in salaries_text_processed_sr]

In [12]:
### Preview bag of words for the randomly selected row
rand_row = random.randint(0,len(bow_corpus)-1)
print("### ROW", rand_row, "###")
print("VECTOR -->", salaries_text_processed_sr[rand_row])
bow_doc_selected = bow_corpus[rand_row]
print("BoW -->", bow_doc_selected)
for i in range(len(bow_doc_selected)):
    print("Word {} (\"{}\") appears {} {}.".format(
        bow_doc_selected[i][0], dictionary[bow_doc_selected[i][0]], bow_doc_selected[i][1], 
        "times" if bow_doc_selected[i][1] > 1 else "time"))

### ROW 12154 ###
VECTOR --> ['it', 'support', 'engineer', 'amazon', 'active', 'directory', 'business', 'analysis', 'computer', 'networks', 'customer', 'service', 'deadline', 'management', 'disaster', 'recovery', 'financial', 'management', 'hardware', 'knowledge', 'hardware', 'repair', 'management', 'microsoft', 'office', 'network', 'configuration', 'network', 'security', 'networking', 'office', 'management', 'process', 'improvement', 'project', 'management', 'records', 'management', 'requirement', 'analysis', 'requirements', 'gathering', 'security', 'strategic', 'planning', 'team', 'building', 'troubleshooting', 'vmware', 'vpn', 'configuration', 'windows', 'windows', 'server', 'active', 'directory', 'business', 'analysis', 'customer', 'service', 'disaster', 'recovery', 'hardware', 'management', 'mediation', 'microsoft', 'office', 'network', 'security', 'networking', 'process', 'improvement', 'project', 'management', 'requirements', 'analysis', 'security', 'strategic', 'planning', 'tea

## Run LDA

In [13]:
# Train the LDA model with BoW using the related method from gensim and save it to 'lda_model_bow'
s_time = time.time()
# lda_model_bow = gensim.models.LdaModel(bow_corpus, num_topics=num_topics, id2word=dictionary, passes=3, random_state=42)
lda_model_bow = gensim.models.LdaModel(corpus=bow_corpus, num_topics=num_topics, id2word=dictionary, distributed=False, 
                  chunksize=2000, passes=3, update_every=1, alpha='asymmetric', eta='auto', decay=0.5, 
                  offset=1.0, eval_every=10, iterations=25, gamma_threshold=0.1, minimum_probability=0.01,
                  minimum_phi_value=0.01, per_word_topics=False, random_state=42)
print("NUMBER OF TOPICS: {} | Time taken for LDA using BoW: {:.4f} seconds.".format(num_topics, time.time() - s_time))

NUMBER OF TOPICS: 25 | Time taken for LDA using BoW: 9.4016 seconds.


In [14]:
# For each topic, explore the words occurring in that topic and its relative weights
# print(dir(lda_model_bow))
print("TOPIC --> WORDS")
for idx, topic in lda_model_bow.print_topics(-1):
    print('{:5} --> {}'.format(idx, topic))

TOPIC --> WORDS
    0 --> 0.142*"sales" + 0.083*"customer" + 0.058*"service" + 0.058*"retail" + 0.047*"management" + 0.047*"inventory" + 0.032*"support" + 0.024*"techniques" + 0.021*"trends" + 0.019*"experience"
    1 --> 0.299*"management" + 0.067*"financial" + 0.045*"office" + 0.039*"microsoft" + 0.032*"planning" + 0.031*"leadership" + 0.029*"records" + 0.028*"deadline" + 0.023*"project" + 0.022*"service"
    2 --> 0.057*"engineering" + 0.053*"maintenance" + 0.050*"electrical" + 0.049*"manufacturing" + 0.034*"power" + 0.031*"systems" + 0.030*"automotive" + 0.026*"troubleshooting" + 0.025*"energy" + 0.021*"lean"
    3 --> 0.112*"business" + 0.090*"management" + 0.082*"analysis" + 0.068*"process" + 0.047*"improvement" + 0.038*"requirements" + 0.026*"project" + 0.020*"software" + 0.019*"development" + 0.019*"gathering"
    4 --> 0.075*"communication" + 0.072*"skills" + 0.055*"problem" + 0.055*"solving" + 0.045*"management" + 0.038*"leadership" + 0.026*"analytical" + 0.026*"teamwork" + 0

## Test LDA with BoW

In [15]:
rand_row = random.randint(0,len(bow_corpus)-1)
print("CHOSEN ROW: {}".format(rand_row))
print("VECTOR >>> {}".format(salaries_text_processed_sr[rand_row]))
print("BAG OF WORDS >>> {}".format(bow_corpus[rand_row]))
print("---------- BoW ----------")
scores_bow = sorted(lda_model_bow[bow_corpus[rand_row]], key=lambda tup: -1*tup[1])
print("SCORE      |  TOPIC")
for index, score in scores_bow:
    print("{:.7f}  |  {:2} --> {}".format(score, index, lda_model_bow.print_topic(index, 5)))

CHOSEN ROW: 15508
VECTOR >>> ['senior', 'data', 'scientist', 'machine', 'learning', 'engineer', 'nlp', 'predictive', 'models', 'sentiment', 'business', 'analysis', 'clinical', 'skills', 'data', 'analysis', 'grant', 'writing', 'marketing', 'research', 'matlab', 'microsoft', 'office', 'public', 'health', 'management', 'research', 'sas', 'spss', 'statistics', 'systems', 'analysis', 'teaching', 'skills', 'writing', 'analysis', 'clinical', 'research', 'data', 'analysis', 'grant', 'writing', 'healthcare', 'life', 'sciences', 'matlab', 'microsoft', 'office', 'public', 'health', 'research', 'sas', 'spss', 'statistics', 'teaching']
BAG OF WORDS >>> [(6, 3), (16, 2), (18, 2), (40, 1), (41, 1), (48, 2), (55, 5), (66, 1), (111, 1), (128, 4), (132, 1), (170, 1), (171, 1), (190, 2), (196, 2), (197, 1), (207, 2), (226, 2), (275, 1), (365, 2), (369, 3), (382, 2), (417, 1), (571, 2), (572, 1), (573, 2), (597, 1), (602, 1), (764, 2), (900, 1), (1398, 1), (1733, 1)]
---------- BoW ----------
SCORE      |

## Assign Topic Numbers For All Rows

In [16]:
classified_rows_bow = {};   topic_model_no = [];   s_time = time.time();
for i in range(num_topics):
    classified_rows_bow[str(i)] = []
# Bag of Words
for i in range(len(salaries_text_processed_sr)):
    scores = sorted(lda_model_bow[bow_corpus[i]], key=lambda tup: -1*tup[1])
    classified_rows_bow[str(scores[0][0])].append(i)
    topic_model_no.append(scores[0][0])
print("============ BoW ============")
print("=== NUMBER ===|=== AMOUNT ===")
for i in range(num_topics):
    print("|     {:2}      |     {:4}    |".format(i, len(classified_rows_bow[str(i)])))
    # print(i, '-->', len(classified_rows_bow[str(i)]), classified_rows[str(i)])
print("="*29)
print(">>> TIME TAKEN FOR TOPIC ASSIGNMENTS: {:.4f} seconds".format(time.time() - s_time))
### Add the feature of these topic model numbers to the dataset
salaries['topic_model_no'] = topic_model_no

============ BoW ============
=== NUMBER ===|=== AMOUNT ===
|      0      |      453    |
|      1      |      757    |
|      2      |      218    |
|      3      |      606    |
|      4      |     1008    |
|      5      |     1337    |
|      6      |     1012    |
|      7      |      511    |
|      8      |      430    |
|      9      |      349    |
|     10      |      612    |
|     11      |      755    |
|     12      |      696    |
|     13      |      823    |
|     14      |     1865    |
|     15      |      220    |
|     16      |       33    |
|     17      |      757    |
|     18      |     1256    |
|     19      |     1072    |
|     20      |     1917    |
|     21      |      501    |
|     22      |       98    |
|     23      |      852    |
|     24      |     1416    |
>>> TIME TAKEN FOR TOPIC ASSIGNMENTS: 2.9052 seconds


## Save Dictionary & LDA Model

In [17]:
### Save the dictionary to disk storage
dictionary_file = datapath("dict_tm25")
dictionary.save(dictionary_file)
print("Saved dictionary to disk...")

### Save the LDA model to disk storage
lda_model_file = datapath("lda_model_tm25")
lda_model_bow.save(lda_model_file)
print("Saved LDA model to disk...")
### Saved at C:\Users\Doğan Yiğit Yenigün\AppData\Local\Programs\Python\Python310\Lib\site-packages\gensim\test\test_data

Saved dictionary to disk...
Saved LDA model to disk...


In [18]:
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd,country_coef,title_code,title_slr_coef,title_seniority,title_seniority_slr_coef,position_code,position_slr_coef,position_seniority,position_seniority_slr_coef,employment_type_coef,emp_type,employment_type_slr_coef,highest_edu_deg_name,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_code,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_code,company_slr_coef,main_skill_code,main_skill_slr_coef_avg,main_skill_slr_coef_max,topic_model_no
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000,1.00,WC68,0.488,NO_SENIORITY,0.488,WC66,0.492,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,MASTERS,0.75,0.770,EDM17,0.796,0.796,UNCT,0.368,"MS01,MS02,MS03,MS04,MS05,MS06,MS07,MS11,MS17,M...",0.872,0.944,5
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000,1.00,UNCT,0.319,ASSOCIATE,0.319,UNCT,0.329,ASSOCIATE,0.340,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,NO_DEGREE,0.200,0.200,NO_COMPANY,0.200,"MS00,MS13,MS15,MS16,MS28,MS30,MS31,MS33,MS34,M...",0.646,0.731,0
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191,0.61,WC76,0.399,NO_SENIORITY,0.488,WC67,0.400,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,"EDM14,EDM30,EDM38,EDM53",0.779,0.823,NO_COMPANY,0.200,"MS00,MS16,MS40,MS42,MS43,MS45,MS62",0.774,0.844,21
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000,1.00,WC36,0.955,NO_SENIORITY,0.488,WC32,0.959,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM14,0.823,0.823,CMP01,0.845,"MS20,MS42,MS43",0.873,0.944,17
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000,1.00,WC07,0.503,NO_SENIORITY,0.488,WC06,0.516,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM19,0.655,0.655,CMP00,0.593,"MS03,MS08,MS12,MS14,MS19",0.769,0.870,24


## Save dataset with TM

In [19]:
### Save the dataset that includes the assigned topic numbers
# salaries.to_csv('Salaries_v4_202412161100_enhanced_tm.csv')

# ML Training

## Preprocess the Dataset

In [20]:
### Fill the null values with average for some columns
salaries['company_score'] = salaries['company_score'].fillna(salaries['company_score'].mean())
salaries['education_score'] = salaries['education_score'].fillna(salaries['education_score'].mean())

In [21]:
### Select the columns (features) for ML models
salaries = salaries[['company_score','working_year','education_score','ms_counts','skill_counts','country_coef', \
                         'title_slr_coef','title_seniority_slr_coef','position_slr_coef','position_seniority_slr_coef', \
                         'employment_type_coef','employment_type_slr_coef','highest_edu_deg_lvl','edu_deg_slr_coef', \
                         'edu_degree_major_slr_coef_max','company_slr_coef','main_skill_slr_coef_avg','amount_usd']]

## Separate Features & Target

In [22]:
X = salaries.drop('amount_usd',axis=1)
Y = salaries['amount_usd']

In [23]:
X.head()

,company_score,working_year,education_score,ms_counts,skill_counts,country_coef,title_slr_coef,title_seniority_slr_coef,position_slr_coef,position_seniority_slr_coef,employment_type_coef,employment_type_slr_coef,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_slr_coef_max,company_slr_coef,main_skill_slr_coef_avg
0,8.900000,11,8.100000,21,20,1.00,0.488,0.488,0.492,0.518,0.99,1.0,0.75,0.770,0.796,0.368,0.872
1,8.248993,5,8.401484,14,10,1.00,0.319,0.319,0.329,0.340,0.99,1.0,0.50,0.657,0.200,0.200,0.646
2,8.248993,12,7.000000,15,20,0.61,0.399,0.488,0.400,0.518,0.99,1.0,0.50,0.657,0.823,0.200,0.774
3,8.700000,13,8.900000,9,20,1.00,0.955,0.488,0.959,0.518,0.10,0.2,0.75,0.770,0.823,0.845,0.873
4,8.400000,3,8.500000,6,9,1.00,0.503,0.488,0.516,0.518,0.10,0.2,0.75,0.770,0.655,0.593,0.769


In [24]:
Y.head()

0    192000
1     38000
2     66191
3    341000
4    185000
Name: amount_usd, dtype: int64

## Train ML Models

### 1. Linear Regression

In [25]:
linreg = LinearRegression()
linreg.get_params()

{'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False}

In [26]:
s_time = time.time()
linreg.fit(X, Y)
print("TRAINING TIME: {:.4f} seconds".format(time.time()-s_time))

TRAINING TIME: 0.0182 seconds


In [27]:
print("COEF:", linreg.coef_)   # Estimated coefficients for the linear regression problem
print("RANK:", linreg.rank_)   # Rank of matrix X. Only available when X is dense
print("INTERCEPT:", linreg.intercept_)   # Independent term in the linear model. Set to 0.0 if fit_intercept = False
print("SINGULAR:", linreg.singular_)   # Singular values of X. Only available when X is dense
print("FEATURES:", linreg.n_features_in_)   # Number of features seen during fit

COEF: [ 1.73063164e+04  1.81152687e+03  1.40827964e+03 -4.96677625e+01
  5.76291429e+01  1.99309470e+05  1.37390871e+05  2.72965715e+04
  1.29848727e+05  8.21874549e+04  9.68711211e+03 -1.08052797e+04
 -6.44339512e+03  2.80752608e+04 -2.90833477e+03  1.09889331e+05
  6.21576268e+04]
RANK: 17
INTERCEPT: -503671.64894331805
SINGULAR: [1379.7207785  1017.13282157  563.53072837  169.32325465   78.77578028
   70.46119627   36.62951344   27.14017416   21.50977196   16.12288543
   14.87856999   14.1319407     8.95281387    5.76480834    4.81298975
    4.08248916    3.75564853]
FEATURES: 17


In [28]:
with open('ml_linreg_proto.pkl', 'wb') as model_file:
    pickle.dump(linreg, model_file)

### 2. Random Forest Regression

In [29]:
rforest = RandomForestRegressor(n_estimators=50, oob_score=True)
rforest.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 50,
 'n_jobs': None,
 'oob_score': True,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [30]:
s_time = time.time()
rforest.fit(X, Y)
print("TRAINING TIME: {:.4f} seconds".format(time.time()-s_time))

TRAINING TIME: 7.2251 seconds


In [31]:
print("BASE ESTIMATOR:", rforest.base_estimator_)   # Estimator used to grow the ensemble
# print("ESTIMATORS:", rforest.estimators_)   # The collection of fitted sub-estimators
print("FEATURE IMPORTANCES:", rforest.feature_importances_)   # The impurity-based feature importances
print("TOP 8 FEATURE IMPORTANCES:")
print(pd.Series(rforest.feature_importances_, index=salaries.columns[:X.shape[1]]).sort_values(ascending=False)[:8])
# Score of the training dataset obtained using an out-of-bag estimate. Exists only when oob_score = True
print("OOB SCORE:", rforest.oob_score_)
# Prediction computed with out-of-bag estimate on the training set. Exists only when oob_score = True
print("OOB PREDICTION:", rforest.oob_prediction_, "", len(rforest.oob_prediction_))
print("FEATURES:", rforest.n_features_in_)   # Number of features seen during fit
print("OUTPUTS:", rforest.n_outputs_, "\n")   # The number of outputs when fit is performed

BASE ESTIMATOR: DecisionTreeRegressor()
FEATURE IMPORTANCES: [0.03161929 0.05644054 0.02030649 0.02057423 0.01298575 0.1212003
 0.10836226 0.00725926 0.31392186 0.01835003 0.00228741 0.00230898
 0.00337611 0.00346555 0.01728581 0.22528815 0.03496799]
TOP 8 FEATURE IMPORTANCES:
position_slr_coef          0.313922
company_slr_coef           0.225288
country_coef               0.121200
title_slr_coef             0.108362
working_year               0.056441
main_skill_slr_coef_avg    0.034968
company_score              0.031619
ms_counts                  0.020574
dtype: float64
OOB SCORE: 0.8271267391771903
OOB PREDICTION: [308611.11111111  72552.5         71309.15789474 ...  58181.81818182
 169058.82352941 275187.5       ]  19554
FEATURES: 17
OUTPUTS: 1 



In [32]:
with open('ml_rforest_reg_proto.pkl', 'wb') as model_file:
    pickle.dump(rforest, model_file)

### 3. Gradient Boosting Regression

In [33]:
gradBoost = GradientBoostingRegressor()
gradBoost.get_params()

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': None,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [34]:
s_time = time.time()
gradBoost.fit(X, Y)
print("TRAINING TIME: {:.4f} seconds".format(time.time()-s_time))

TRAINING TIME: 2.4947 seconds


In [35]:
print("INIT:", gradBoost.init_)  # The estimator that provides the initial predictions
print("ESTIMATORS (FIRST 5):\n{}".format(gradBoost.estimators_[:5]))  # The collection of fitted sub-estimators
# The number of estimators as selected by early stopping (if 'n_iter_no_change' is specified).
# Otherwise, it is set to 'n_estimators'
print("# OF ESTIMATORS:", gradBoost.n_estimators_)
print("# OF FEATURES:", gradBoost.n_features_in_)  # Number of features seen during fit
print("MAX FEATURES:", gradBoost.max_features_)  # The inferred value of max_features
print("FEATURE IMPORTANCES:", gradBoost.feature_importances_)  # The impurity-based feature importances
# The i'th score 'train_score[i]' is the loss of the model at iteration i on the in-bag sample.
# If subsample == 1, this is the loss on the training data.
print("TRAIN SCORE:\n{}".format(gradBoost.train_score_))

INIT: DummyRegressor()
ESTIMATORS (FIRST 5):
[[DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                        random_state=RandomState(MT19937) at 0x209A8C4ED40)]
 [DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                        random_state=RandomState(MT19937) at 0x209A8C4ED40)]
 [DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                        random_state=RandomState(MT19937) at 0x209A8C4ED40)]
 [DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                        random_state=RandomState(MT19937) at 0x209A8C4ED40)]
 [DecisionTreeRegressor(criterion='friedman_mse', max_depth=3,
                        random_state=RandomState(MT19937) at 0x209A8C4ED40)]]
# OF ESTIMATORS: 100
# OF FEATURES: 17
MAX FEATURES: 17
FEATURE IMPORTANCES: [1.43280225e-02 4.58860318e-02 1.34699968e-03 4.04816441e-05
 1.39194237e-04 1.46089372e-01 1.72405366e-01 5.42123302e-03
 2.59096765e-01 2.58823517e-02 1.20318904e-05 1.4101185

In [36]:
with open('ml_gradboost_reg_proto.pkl', 'wb') as model_file:
    pickle.dump(gradBoost, model_file)

### 4. XGBoost Regression

In [37]:
xgb = xgboost.XGBRegressor()
xgb.get_params()

{'objective': 'reg:squarederror',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [38]:
s_time = time.time()
xgb.fit(X, Y)
print("TRAINING TIME: {:.4f} seconds".format(time.time()-s_time))

TRAINING TIME: 0.1523 seconds


In [39]:
with open('ml_xgboost_reg_proto.pkl', 'wb') as model_file:
    pickle.dump(xgb, model_file)